In [ ]:
# Chargement des packages
import numpy as np
import pandas as pd
%pip install openpyxl

In [ ]:
# Chargement des données
synthese = pd.read_excel("AGRIBALYSE3.2_Tableur produits alimentaires_PublieAOUT25.xlsx", sheet_name="Synthese", header=None)
detail_etape = pd.read_excel("AGRIBALYSE3.2_Tableur produits alimentaires_PublieAOUT25.xlsx", sheet_name="Detail etape", header=None)

Feuillet "Synthese"

Ce feuillet contient 2461 lignes et 32 colonnes.

In [ ]:
# On souhaite créer un nom unique par colonne. 
# Les 12 premières colonnes n'ont qu'un titre.
# Les colonnes suivantes ont un titre et une unité.

# On travaille sur les trois premières lignes.
entetes_synthese = synthese.iloc[:3]

# On crée la liste des noms de colonnes.
noms_colonnes_synthese = []

for i in range(entetes_synthese.shape[1]):

    info1 = entetes_synthese.iloc[1, i] 
    info2 = entetes_synthese.iloc[2, i]

    # Pour les 12 premières colonnes
    if i < 12:
        nom = str(info2).replace("\n", " ").strip() 

    # Pour les colonnes suivantes
    else:
        nom = (
            str(info1).replace("\n", " ").strip()
            + " — "
            + str(info2).replace("\n", " ").strip()
        )

    noms_colonnes_synthese.append(nom)

# On retire les trois lignes d'en-têtes et on attribue les noms aux colonnes.
synthese = synthese.iloc[3:].copy()
synthese.columns = noms_colonnes_synthese

# On réinitialise les index.
synthese = synthese.reset_index(drop=True)

display(synthese.head(3))

Le feuillet possède désormais 2458 lignes.

In [ ]:
# On renomme les variables du feuillet "Synthese".
renommage = {
    "Code AGB": "code_agb",
    "Code CIQUAL": "code_ciqual",
    "Groupe d'aliment": "groupe_aliment",
    "Sous-groupe d'aliment": "sous_groupe_aliment",
    "Nom du Produit en Français": "nom_produit",
    "LCI Name": "lci_name",
    "code saison (0 : hors saison ; 1 : de saison ; 2 : mix de consommation FR)": "code_saison",
    "code avion (1 : par avion)": "code_avion",
    "Livraison": "livraison",
    "Approche emballage": "approche_emballage",
    "Préparation": "preparation",
    "DQR - Note de qualité de la donnée (1 excellente ; 5 très faible)": "dqr",
    "Score unique EF 3.1 — mPt/kg de produit": "score_unique",
    "Changement climatique — kg CO2 eq/kg de produit": "changement_climatique",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit": "appauvrissement_ozone",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit": "rayonnements_ionisants",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit": "formation_photochimique_ozone",
    "Particules fines — disease inc./kg de produit": "particules_fines",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit": "toxicite_humaine_non_cancer",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit": "toxicite_humaine_cancer",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit": "acidification",
    "Eutrophisation eaux douces — kg P eq/kg de produit": "eutrophisation_eau_douce",
    "Eutrophisation marine — kg N eq/kg de produit": "eutrophisation_marine",
    "Eutrophisation terrestre — mol N eq/kg de produit": "eutrophisation_terrestre",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit": "ecotoxicite_eau_douce",
    "Utilisation du sol — Pt/kg de produit": "utilisation_sol",
    "Épuisement des ressources eau — m3 depriv./kg de produit": "epuisement_eau",
    "Épuisement des ressources énergétiques — MJ/kg de produit": "epuisement_energie",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit": "epuisement_mineraux",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit": "changement_climatique_biogeniques",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit": "changement_climatique_fossiles",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit": "changement_climatique_sols"
}

# On remplace les espaces insécables par des espaces normaux.
synthese.columns = synthese.columns.str.replace("\xa0", " ", regex=False)

# On renomme les colonnes.
synthese = synthese.rename(columns=renommage)

display(synthese.head(3))

Feuillet "Detail etape"

In [ ]:
# On ne conserve que les lignes non entièrement vides à savoir les lignes 0 à 2462 incluse.
print("Nombre de lignes avant nettoyage :", len(detail_etape))
detail_etape = detail_etape.iloc[:2462].reset_index(drop=True)

Le feuillet possède 2462 lignes et 153 colonnes.

In [ ]:
# On souhaite créer un nom unique par colonne. 
# Les 8 premières colonnes du feuillet "Detail etape" n'ont qu'un titre.
# Les colonnes suivantes ont un titre, une unité et une étape (par exemple, agriculture ou transformation).
# Les 5 dernières colonnes n'ont que deux titres.

# On travaille sur les quatre premières lignes.
entetes_detail = detail_etape.iloc[:4].copy()

# On crée la liste des noms de colonnes.
noms_colonnes_detail = []

# On crée la variable indicateur_precedent qui garde le nom du dernier indicateur environnemental trouvé.
indicateur_precedent = ""

# On crée la variable unite_precedente qui garde le nom de la dernière unité trouvée.
unite_precedente = ""

for i in range(entetes_detail.shape[1]):

    # Les 8 premières colonnes ont leur nom sur la ligne 3
    if i < 8:
        nom = str(entetes_detail.iloc[3, i]).replace("\n", " ").strip()

    else:
        info1 = entetes_detail.iloc[1, i]
        info2 = entetes_detail.iloc[2, i]
        info3 = entetes_detail.iloc[3, i]

        # On met à jour l'indicateur si la cellule n'est pas vide
        if pd.notna(info1) and str(info1).strip():
            indicateur_precedent = str(info1).replace("\n", " ").strip()

        # On met à jour l'unité si elle est renseignée
        if pd.notna(info2) and str(info2).strip():
            unite_precedente = str(info2).replace("\n", " ").strip()
        elif "DQR" in str(info1): # pour les 5 dernières colonnes qui contiennent DQR dans leur titre il n'y a pas d'unité
            unite_precedente = ""

        # On récupère le nom situé sur la ligne 3
        sous_colonne = (
            str(info3).replace("\n", " ").strip()
            if pd.notna(info3) else ""
        )

        # On combine les trois informations pour créer le nom final de chaque colonne 
        morceaux = [
            morceau for morceau in
            [indicateur_precedent, unite_precedente, sous_colonne]
            if morceau
        ]
        nom = " — ".join(morceaux)

    noms_colonnes_detail.append(nom)

# On retire les 4 lignes d'en-têtes et on attribue les noms aux colonnes.
detail_etape = detail_etape.iloc[4:].copy()

detail_etape.columns = noms_colonnes_detail

detail_etape.columns = (
    detail_etape.columns
    .str.replace("\xa0", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)
detail_etape = detail_etape.reset_index(drop=True)

display(detail_etape.head(3))

In [ ]:
# On renomme les variables du feuillet "Detail etape".
renommage_detail = {
    # Les 8 premières colonnes
    "Code AGB": "code_agb",
    "Code CIQUAL": "code_ciqual",
    "Groupe d'aliment": "groupe_aliment",
    "Sous-groupe d'aliment": "sous_groupe_aliment",
    "Nom du Produit en Français": "nom_produit",
    "LCI Name": "lci_name",
    "DQR": "dqr",
    "Nom et code": "nom_et_code",

    # Score unique
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Agriculture": "score_unique_agriculture",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Transformation": "score_unique_transformation",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Emballage": "score_unique_emballage",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Transport": "score_unique_transport",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Supermarché et distribution": "score_unique_supermarche_distribution",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Consommation": "score_unique_consommation",
    "Score unique EF 3.1 EF Method 3.1 - implémentation SimaPro (adapted) V1.00 / EF3.1 normalization and weighting set — mPt / kg de produit — Total": "score_unique_total",

    # Changement climatique
    "Changement climatique — kg CO2 eq/kg de produit — Agriculture": "changement_climatique_agriculture",
    "Changement climatique — kg CO2 eq/kg de produit — Transformation": "changement_climatique_transformation",
    "Changement climatique — kg CO2 eq/kg de produit — Emballage": "changement_climatique_emballage",
    "Changement climatique — kg CO2 eq/kg de produit — Transport": "changement_climatique_transport",
    "Changement climatique — kg CO2 eq/kg de produit — Supermarché et distribution": "changement_climatique_supermarche_distribution",
    "Changement climatique — kg CO2 eq/kg de produit — Consommation": "changement_climatique_consommation",
    "Changement climatique — kg CO2 eq/kg de produit — Total": "changement_climatique_total",

    # Appauvrissement de la couche d'ozone
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Agriculture": "appauvrissement_ozone_agriculture",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Transformation": "appauvrissement_ozone_transformation",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Emballage": "appauvrissement_ozone_emballage",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Transport": "appauvrissement_ozone_transport",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Supermarché et distribution": "appauvrissement_ozone_supermarche_distribution",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Consommation": "appauvrissement_ozone_consommation",
    "Appauvrissement de la couche d'ozone — kg CVC11 eq/kg de produit — Total": "appauvrissement_ozone_total",

    # Rayonnements ionisants
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Agriculture": "rayonnements_ionisants_agriculture",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Transformation": "rayonnements_ionisants_transformation",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Emballage": "rayonnements_ionisants_emballage",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Transport": "rayonnements_ionisants_transport",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Supermarché et distribution": "rayonnements_ionisants_supermarche_distribution",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Consommation": "rayonnements_ionisants_consommation",
    "Rayonnements ionisants — kBq U-235 eq/kg de produit — Total": "rayonnements_ionisants_total",

    # Formation photochimique d'ozone
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Agriculture": "formation_photochimique_ozone_agriculture",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Transformation": "formation_photochimique_ozone_transformation",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Emballage": "formation_photochimique_ozone_emballage",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Transport": "formation_photochimique_ozone_transport",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Supermarché et distribution": "formation_photochimique_ozone_supermarche_distribution",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Consommation": "formation_photochimique_ozone_consommation",
    "Formation photochimique d'ozone — kg NMVOC eq/kg de produit — Total": "formation_photochimique_ozone_total",

    # Particules fines
    "Particules fines — disease inc./kg de produit — Agriculture": "particules_fines_agriculture",
    "Particules fines — disease inc./kg de produit — Transformation": "particules_fines_transformation",
    "Particules fines — disease inc./kg de produit — Emballage": "particules_fines_emballage",
    "Particules fines — disease inc./kg de produit — Transport": "particules_fines_transport",
    "Particules fines — disease inc./kg de produit — Supermarché et distribution": "particules_fines_supermarche_distribution",
    "Particules fines — disease inc./kg de produit — Consommation": "particules_fines_consommation",
    "Particules fines — disease inc./kg de produit — Total": "particules_fines_total",

    # Toxicité humaine non cancérogène
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Agriculture": "toxicite_humaine_non_cancer_agriculture",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Transformation": "toxicite_humaine_non_cancer_transformation",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Emballage": "toxicite_humaine_non_cancer_emballage",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Transport": "toxicite_humaine_non_cancer_transport",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Supermarché et distribution": "toxicite_humaine_non_cancer_supermarche_distribution",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Consommation": "toxicite_humaine_non_cancer_consommation",
    "Effets toxicologiques sur la santé humaine : substances non-cancérogènes — CTUh/kg de produit — Total": "toxicite_humaine_non_cancer_total",

    # Toxicité humaine cancérogène
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Agriculture": "toxicite_humaine_cancer_agriculture",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Transformation": "toxicite_humaine_cancer_transformation",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Emballage": "toxicite_humaine_cancer_emballage",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Transport": "toxicite_humaine_cancer_transport",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Supermarché et distribution": "toxicite_humaine_cancer_supermarche_distribution",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Consommation": "toxicite_humaine_cancer_consommation",
    "Effets toxicologiques sur la santé humaine : substances cancérogènes — CTUh/kg de produit — Total": "toxicite_humaine_cancer_total",

    # Acidification
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Agriculture": "acidification_agriculture",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Transformation": "acidification_transformation",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Emballage": "acidification_emballage",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Transport": "acidification_transport",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Supermarché et distribution": "acidification_supermarche_distribution",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Consommation": "acidification_consommation",
    "Acidification terrestre et eaux douces — mol H+ eq/kg de produit — Total": "acidification_total",

    # Eutrophisation eaux douces
    "Eutrophisation eaux douces — kg P eq/kg de produit — Agriculture": "eutrophisation_eau_douce_agriculture",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Transformation": "eutrophisation_eau_douce_transformation",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Emballage": "eutrophisation_eau_douce_emballage",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Transport": "eutrophisation_eau_douce_transport",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Supermarché et distribution": "eutrophisation_eau_douce_supermarche_distribution",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Consommation": "eutrophisation_eau_douce_consommation",
    "Eutrophisation eaux douces — kg P eq/kg de produit — Total": "eutrophisation_eau_douce_total",

    # Eutrophisation marine
    "Eutrophisation marine — kg N eq/kg de produit — Agriculture": "eutrophisation_marine_agriculture",
    "Eutrophisation marine — kg N eq/kg de produit — Transformation": "eutrophisation_marine_transformation",
    "Eutrophisation marine — kg N eq/kg de produit — Emballage": "eutrophisation_marine_emballage",
    "Eutrophisation marine — kg N eq/kg de produit — Transport": "eutrophisation_marine_transport",
    "Eutrophisation marine — kg N eq/kg de produit — Supermarché et distribution": "eutrophisation_marine_supermarche_distribution",
    "Eutrophisation marine — kg N eq/kg de produit — Consommation": "eutrophisation_marine_consommation",
    "Eutrophisation marine — kg N eq/kg de produit — Total": "eutrophisation_marine_total",

    # Eutrophisation terrestre
    "Eutrophisation terrestre — mol N eq/kg de produit — Agriculture": "eutrophisation_terrestre_agriculture",
    "Eutrophisation terrestre — mol N eq/kg de produit — Transformation": "eutrophisation_terrestre_transformation",
    "Eutrophisation terrestre — mol N eq/kg de produit — Emballage": "eutrophisation_terrestre_emballage",
    "Eutrophisation terrestre — mol N eq/kg de produit — Transport": "eutrophisation_terrestre_transport",
    "Eutrophisation terrestre — mol N eq/kg de produit — Supermarché et distribution": "eutrophisation_terrestre_supermarche_distribution",
    "Eutrophisation terrestre — mol N eq/kg de produit — Consommation": "eutrophisation_terrestre_consommation",
    "Eutrophisation terrestre — mol N eq/kg de produit — Total": "eutrophisation_terrestre_total",

    # Écotoxicité eau douce
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Agriculture": "ecotoxicite_eau_douce_agriculture",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Transformation": "ecotoxicite_eau_douce_transformation",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Emballage": "ecotoxicite_eau_douce_emballage",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Transport": "ecotoxicite_eau_douce_transport",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Supermarché et distribution": "ecotoxicite_eau_douce_supermarche_distribution",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Consommation": "ecotoxicite_eau_douce_consommation",
    "Écotoxicité pour écosystèmes aquatiques d'eau douce — CTUe/kg de produit — Total": "ecotoxicite_eau_douce_total",

    # Utilisation du sol
    "Utilisation du sol — Pt/kg de produit — Agriculture": "utilisation_sol_agriculture",
    "Utilisation du sol — Pt/kg de produit — Transformation": "utilisation_sol_transformation",
    "Utilisation du sol — Pt/kg de produit — Emballage": "utilisation_sol_emballage",
    "Utilisation du sol — Pt/kg de produit — Transport": "utilisation_sol_transport",
    "Utilisation du sol — Pt/kg de produit — Supermarché et distribution": "utilisation_sol_supermarche_distribution",
    "Utilisation du sol — Pt/kg de produit — Consommation": "utilisation_sol_consommation",
    "Utilisation du sol — Pt/kg de produit — Total": "utilisation_sol_total",

    # Épuisement des ressources en eau
    "Épuisement des ressources eau — m3 depriv./kg de produit — Agriculture": "epuisement_eau_agriculture",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Transformation": "epuisement_eau_transformation",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Emballage": "epuisement_eau_emballage",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Transport": "epuisement_eau_transport",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Supermarché et distribution": "epuisement_eau_supermarche_distribution",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Consommation": "epuisement_eau_consommation",
    "Épuisement des ressources eau — m3 depriv./kg de produit — Total": "epuisement_eau_total",

    # Épuisement des ressources énergétiques
    "Épuisement des ressources énergétiques — MJ/kg de produit — Agriculture": "epuisement_energie_agriculture",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Transformation": "epuisement_energie_transformation",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Emballage": "epuisement_energie_emballage",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Transport": "epuisement_energie_transport",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Supermarché et distribution": "epuisement_energie_supermarche_distribution",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Consommation": "epuisement_energie_consommation",
    "Épuisement des ressources énergétiques — MJ/kg de produit — Total": "epuisement_energie_total",

    # Épuisement des ressources minérales
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Agriculture": "epuisement_mineraux_agriculture",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Transformation": "epuisement_mineraux_transformation",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Emballage": "epuisement_mineraux_emballage",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Transport": "epuisement_mineraux_transport",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Supermarché et distribution": "epuisement_mineraux_supermarche_distribution",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Consommation": "epuisement_mineraux_consommation",
    "Épuisement des ressources minéraux — kg Sb eq/kg de produit — Total": "epuisement_mineraux_total",

    # Émissions biogéniques
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Agriculture": "changement_climatique_biogeniques_agriculture",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Transformation": "changement_climatique_biogeniques_transformation",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Emballage": "changement_climatique_biogeniques_emballage",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Transport": "changement_climatique_biogeniques_transport",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Supermarché et distribution": "changement_climatique_biogeniques_supermarche_distribution",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Consommation": "changement_climatique_biogeniques_consommation",
    "Changement climatique - émissions biogéniques — kg CO2 eq/kg de produit — Total": "changement_climatique_biogeniques_total",

    # Émissions fossiles
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Agriculture": "changement_climatique_fossiles_agriculture",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Transformation": "changement_climatique_fossiles_transformation",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Emballage": "changement_climatique_fossiles_emballage",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Transport": "changement_climatique_fossiles_transport",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Supermarché et distribution": "changement_climatique_fossiles_supermarche_distribution",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Consommation": "changement_climatique_fossiles_consommation",
    "Changement climatique - émissions fossiles — kg CO2 eq/kg de produit — Total": "changement_climatique_fossiles_total",

    # Émissions liées au changement d'affectation des sols
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Agriculture": "changement_climatique_sols_agriculture",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Transformation": "changement_climatique_sols_transformation",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Emballage": "changement_climatique_sols_emballage",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Transport": "changement_climatique_sols_transport",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Supermarché et distribution": "changement_climatique_sols_supermarche_distribution",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Consommation": "changement_climatique_sols_consommation",
    "Changement climatique - émissions liées au changement d'affectation des sols — kg CO2 eq/kg de produit — Total": "changement_climatique_sols_total",

    # DQR détaillé
    "Detailed DQR of the prodct —  — DQR Overall": "dqr_detail_overall",
    "Detailed DQR of the prodct —  — P": "dqr_detail_p",
    "Detailed DQR of the prodct —  — TiR": "dqr_detail_tir",
    "Detailed DQR of the prodct —  — GR": "dqr_detail_gr",
    "Detailed DQR of the prodct —  — TeR": "dqr_detail_ter"
}

# On nettoie les espaces.
detail_etape.columns = detail_etape.columns.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# On nettoie les espaces dans renommage_detail.
renommage_detail = {
    " ".join(ancien.replace("\xa0", " ").split()): nouveau
    for ancien, nouveau in renommage_detail.items()
}

# On renomme les colonnes.
detail_etape = detail_etape.rename(columns=renommage_detail)

display(detail_etape.head(3))

Merge des deux feuillets

In [ ]:
# Nombre de codes CIQUAL uniques

print("Nombre de codes CIQUAL uniques dans synthese:", synthese["code_ciqual"].nunique())
print("Nombre de codes CIQUAL répétés dans synthese:", synthese["code_ciqual"].duplicated().sum())

print("Nombre de codes CIQUAL uniques dans detail etape:", detail_etape["code_ciqual"].nunique())
print("Nombre de codes CIQUAL répétés dans detail etape:", detail_etape["code_ciqual"].duplicated().sum())

Dans chacun des deux feuillets, il y a 2425 codes CIQUAL distincts et 33 qui apparaissent deux fois. 

In [ ]:
# Nombre de codes AGB uniques

print("Nombre de codes AGB uniques dans synthese:", synthese["code_agb"].nunique())
print("Nombre de codes AGB répétés dans synthese:", synthese["code_agb"].duplicated().sum())

print("Nombre de codes AGB uniques dans detail etape:", detail_etape["code_agb"].nunique())
print("Nombre de codes AGB répétés dans detail etape:", detail_etape["code_agb"].duplicated().sum())

Dans chacun des deux feuillets, il y a 2 451 codes AGB distincts et 7 qui apparaissent deux fois, mais sont associés à des noms de produits différents. 

In [ ]:
# On affiche les lignes dont le code AGB apparaît plusieurs fois.
print("Doublons dans Synthese :")
display(
    synthese[synthese["code_agb"].duplicated(keep=False)]
    .sort_values("code_agb")
)

print("Doublons dans Detail etape :")
display(
    detail_etape[detail_etape["code_agb"].duplicated(keep=False)]
    .sort_values("code_agb")
)

In [ ]:
# On compare les deux feuillets sur les colonnes code_agb, code_ciqual et nom_produit.
comparaison = pd.merge(
    synthese[["code_agb", "code_ciqual", "nom_produit"]],
    detail_etape[["code_agb", "code_ciqual", "nom_produit"]],
    on=["code_agb", "code_ciqual", "nom_produit"],
    how="outer",
    indicator=True
)

print(comparaison["_merge"].value_counts())
non_correspondances = comparaison[comparaison["_merge"] != "both"]
display(non_correspondances.sort_values(["code_agb", "_merge"]))

La comparaison des deux feuillets donne 2 448 correspondances avec le même code AGB, le même code CIQUAL et le même nom de produit. En revanche, 20 lignes ne correspondent pas exactement (10 dans chaque feuillet).